# 25 - FrozenLake DQN with Reasoning

Choose where to apply the value loss independently of the reasoning recurrence.

- `REASONING_LOSS = "all_tokens"` trains a Bellman loss at every generated
  reasoning state. A shared head predicts environment actions plus `think`.
  Environment actions are unavailable before `MIN_REASONING_STEPS`; think is
  unavailable at `MAX_REASONING_STEPS`.
- `REASONING_LOSS = "last_token"` trains only the final state's recorded
  environment action. Set the minimum equal to the maximum for a fixed budget.
  The head has only environment actions, with no learned stopping decision.

In either mode, `RESIDUAL_CONNECTIONS = True` accumulates
`z_next = z + update_projection(update_norm(transformer_output))` and feeds normalized detached residual states
into successive positions. A dedicated trainable start embedding initializes each decision. The extra residual path carries gradients through
the accumulated updates. `RESIDUAL_CONNECTIONS = False` instead repeats the
**same trainable start embedding** at every causal position; its transformer
output predicts that step's values and is never fed back as the next token.
Later tokens can still attend to earlier positions in the transformer history.

The first step is free. All-token supervision charges a cost for each `think`
transition; last-token supervision subtracts the total cost `c * (K - 1)` once.
Reported environment rewards exclude these internal costs. The backbone has no
final output normalization. Generation always uses continuous paged KV caching.

The shared update RMSNorm is non-affine (epsilon required from backbone.rms_norm_eps; Qwen3 uses 1e-6). Its H-to-H linear map starts with zero weights and zero bias.


In [ ]:
import json
from pathlib import Path
from tempfile import TemporaryDirectory
from typing import Literal, cast

import torch
from huggingface_hub import HfApi
from tensordict import TensorDict
from torch import nn
from transformers import AutoTokenizer

from mouse_core import MouseOptimizer
from mouse_core.data import (
    DataLoader,
    EnvironmentProtocol,
    Tokenizer,
    compose,
    environment_decisions,
    load_stores_from_hub,
    save_tokenizer,
)
from mouse_core.models import (
    Backbone,
    BaseBackbone,
    Embedding,
    Model,
    MultiEmbedding,
    ReasoningBatch,
    SequentialReasoner,
    save_model,
)
from mouse_core.models.backbone.continuous_decode import ContinuousDecodeSession
from mouse_core.objectives import DqnObjective, boundary_discount, final_reasoning_loss, value_reasoning_loss

# Repositories
DATASET_ID = "mouse-example-frozenlake-reasoning-dataset"
MODEL_ID = "mouse-example-reasoning-dqn"

# Environment
IS_SLIPPERY = False
MAX_EPISODE_STEPS = 16

# Model
PRETRAINED = "Qwen/Qwen3-0.6B"

# Tokenizer
GENERATION_START_TOKEN = "\n"  # Existing single-token newline; no vocabulary expansion.

# Reasoning
REASONING_LOSS: Literal["all_tokens", "last_token"] = "all_tokens"
RESIDUAL_CONNECTIONS = True
MIN_REASONING_STEPS = 1
MAX_REASONING_STEPS = 16
REASONING_STEP_COST = 0.01
MAX_ACTIVE_REQUESTS = 8

# Training
PREFETCH = 4
SAMPLE_BUDGET = 8
NUM_WORKERS = 0
NUM_CYCLES = 1
TRAIN_STEPS = 10

# Objectives
GAMMA = 0.99
GAMMA_THINK = 1.0

# Optimizer
LR_EMBEDDING = 1e-5
LR_BACKBONE = 1e-5
LR_HEADS = 1e-5
MAX_GRAD_NORM = 1.0

# Target updates
POLYAK_TAU_EMBEDDING = 0.1
POLYAK_TAU_BACKBONE = 0.1
POLYAK_TAU_VALUE = 0.001

# Random seeds
SEED = 7

# Runtime options
USE_KERNELS = False
GRADIENT_CHECKPOINTING = True

## Runtime


In [ ]:
if not torch.cuda.is_available():
    raise RuntimeError(
        "This notebook trains with FlexAttention and requires CUDA; CPU supports inference only."
    )
device = torch.device("cuda")
torch.manual_seed(SEED)

## Load Data

This is offline RL. Load existing Mouse event stores; this notebook never creates
or steps an environment. Set `DATASET_ID` to a previously recorded dataset of
standalone 4x4 `FrozenLake-v1` episodes, with actions 0=left, 1=down, 2=right, 3=up.
Set `IS_SLIPPERY` and `MAX_EPISODE_STEPS` to match that dataset. The dataset ID is
an example name to replace with your own source. Use the collector's
coordinator/agent/environment event schema with `_episode_start`, `_terminated`,
and `_truncated`; repeated procedural-map tasks require their own matching prompt.

Only complete episodes are sampled. Arrival rows hold the recorded previous
action and reward. Terminal and time-limit rows have no value readout. Generating
reasoning adds model computation over these recorded histories, not interactions.
Evaluate saved checkpoints separately in [15 — Inference](15_inference.ipynb).


In [ ]:
stores = load_stores_from_hub(repo_id=DATASET_ID, split="train")

## Tokenizer

Render each observed arrival as a prompt event. The same tokenizer defines text
inputs, the generation-start marker, and objective labels. The initial event uses
the chat template; later events append the incoming action, reward, and observation.
Terminal observations retain their labels but have no prediction marker.

In [ ]:
def task_instructions() -> str:
    """Save the exact inference prompt alongside the checkpoint."""
    return (
        "Estimate action values in FrozenLake: a 4 by 4 grid, with states numbered row-major 0 to 15. "
        "Actions: 0=left, 1=down, 2=right, 3=up. The map is hidden. "
        "The goal rewards 1; a hole ends the episode with reward 0. "
        f"The episode ends after at most {MAX_EPISODE_STEPS} steps. "
        f"Slippery transitions: {IS_SLIPPERY}. "
        "Estimate values from the observed history.\nHistory:\n"
    )


def format_observation(*, observation, action, reward, requested):
    """Render one arrival; the tokenizer adds its action-request marker."""
    if action is None:
        text = text_tokenizer.apply_chat_template(
            [
                {
                    "role": "user",
                    "content": task_instructions() + f"State {observation}.\n",
                }
            ],
            tokenize=False,
            add_generation_prompt=True,
            enable_thinking=False,
        )
    else:
        text = f"Action {action}, reward {reward:g}, state {observation}.\n"
    return text + ("Value:" if requested else "")


def format_events(events):
    """Convert sampled source events to prompt events; the loader owns packing."""
    protocol = EnvironmentProtocol(
        episode_start="_episode_start", terminated="_terminated", truncated="_truncated"
    )
    formatted = []
    for row in environment_decisions(events, protocol=protocol):
        observation, action = row["observation"], row.get("action")
        if row.get("inner_terminated") or row.get("inner_truncated"):
            raise ValueError(
                "Use standalone 4x4 FrozenLake episodes, without inner repeats"
            )
        if not 0 <= observation < 16 or (action is not None and not 0 <= action < 4):
            raise ValueError(
                "This example expects 4x4 FrozenLake observations and four actions"
            )
        reward = float(row.get("reward", 0.0))
        terminated = bool(row.get("_terminated", False))
        truncated = bool(row.get("_truncated", False))
        requested = bool(row.get("action_requested", False))
        if not (terminated or truncated) and not requested:
            raise ValueError("Each nonterminal observation must request an action")
        payload = {
            "prompt": format_observation(
                observation=int(observation),
                action=action,
                reward=reward,
                requested=requested,
            ),
            "reward": reward,
            "terminated": terminated,
            "truncated": truncated,
        }
        if action is not None:
            payload["action"] = int(action)
        if requested:
            payload["action_requested"] = True
        formatted.append(
            {
                "provider": "environment",
                "episode_index": row["episode_index"],
                "step_index": row["step_index"],
                "payload": payload,
            }
        )
    return formatted


text_tokenizer = AutoTokenizer.from_pretrained(PRETRAINED)
tokenizer = Tokenizer(
    tokenizers={"text": text_tokenizer},
    step_field="step_index",
    input_fields=[
        {
            "type": "text",
            "tokenizer": "text",
            "provider": "environment",
            "input_field": "prompt",
            "format": "{prompt}",
        },
        {
            "type": "token",
            "tokenizer": "text",
            "provider": "environment",
            "input_field": "action_requested",
            "format": GENERATION_START_TOKEN,
            "head_output": True,
            "autoregressive": True,
        },
    ],
    objective_fields=[
        {
            "input_field": "action",
            "missing_value": None,
        },
        {
            "input_field": "reward",
            "missing_value": 0.0,
        },
        {
            "input_field": "terminated",
            "missing_value": False,
        },
        {
            "input_field": "truncated",
            "missing_value": False,
        },
    ],
)
# Fail before model allocation if the configured marker is not a single token.
tokenizer({"provider": "environment", "payload": {"action_requested": True}})

## Replay

Use the standard `DataLoader` to sample complete recorded episodes. The transform
formats source events and calls `tokenizer.encode`; the loader packs tokens,
objective labels, sequence IDs, and prediction-to-step mappings. `SAMPLE_BUDGET`
fixes the number of complete outer episodes per batch at eight, including all inner
repeats. Generated reasoning has its own cap and does not reduce the sample count.

No environment interactions occur here. Each optimizer update consumes
`loader.next_batch()`, generates fresh reasoning at every action request, then trains.

In [ ]:
train_transform = compose(stages=(format_events, tokenizer.encode))


def sample_ranges(dataset):
    """Keep all events belonging to one outer episode together."""
    start, previous = 0, None
    for index, episode in enumerate(dataset["episode_index"][:]):
        if episode is not None and episode != previous:
            if index:
                yield start, index
            start, previous = index, episode
    if len(dataset):
        yield start, len(dataset)


def sample_eligible(events):
    """Exclude incomplete prefixes and unfinished episode tails."""
    first, last = events[0], events[-1]
    return (
        first["provider"] == "coordinator"
        and first["step_index"] == 0
        and first["payload"].get("_episode_start") is True
        and last["provider"] == "environment"
        and (
            last["payload"].get("_terminated") is True
            or last["payload"].get("_truncated") is True
        )
    )


loader = DataLoader(
    stores=stores,
    sample_budget=SAMPLE_BUDGET,
    sample_ranges=sample_ranges,
    sample_eligible=sample_eligible,
    transform=train_transform,
    prefetch=PREFETCH,
    num_workers=NUM_WORKERS,
    seed=SEED,
)

## Build Encoder and Heads

`REASONING_LOSS` selects an A+1 action head with adaptive depth or an A-action
head with a fixed budget. `RESIDUAL_CONNECTIONS` independently controls the
extra residual update. `SequentialReasoner` owns the trainable input RMSNorm
and a shared H-to-H linear update projection, initialized to identity.
Online and delayed models replay the same generated history. Only online replay
receives gradients; eval mode disables dropout through backward.


In [ ]:
class ReasoningDqn(Model):
    def __init__(
        self,
        *,
        embedding: MultiEmbedding,
        backbone: BaseBackbone,
        value_head: nn.Module,
        min_steps: int,
        max_steps: int,
        residual_connections: bool,
        reasoning_loss: Literal["all_tokens", "last_token"],
    ) -> None:
        super().__init__()
        if reasoning_loss not in {"all_tokens", "last_token"}:
            raise ValueError("reasoning_loss must be all_tokens or last_token")
        if reasoning_loss == "last_token" and min_steps != max_steps:
            raise ValueError("last_token requires equal minimum and maximum reasoning steps")
        self.reasoning_loss = reasoning_loss
        # Both context and starting token embeddings train during forward.
        self.embedding = embedding
        self.backbone = backbone
        self.value_head = value_head
        self.reasoner = SequentialReasoner(
            hidden_dim=backbone.hidden_dim, min_steps=min_steps, max_steps=max_steps,
            residual_connections=residual_connections, adaptive_depth=reasoning_loss == "all_tokens",
         rms_norm_eps=backbone.rms_norm_eps)

    @torch.no_grad()
    def generate(
        self,
        tokens: TensorDict,
        *,
        max_active_requests: int = 8,
        cache: ContinuousDecodeSession | None = None,
    ) -> ReasoningBatch:
        return self.reasoner.generate(
            tokens,
            embedding=self.embedding,
            backbone=self.backbone,
            action_values=self.value_head,
            backend="continuous",
            max_active_requests=max_active_requests,
            cache=cache,
        )

    def forward(self, generated: ReasoningBatch) -> dict[str, torch.Tensor]:
        states = self.reasoner(
            generated,
            embedding=self.embedding,
            backbone=self.backbone,
        )
        rows = torch.arange(len(states), device=states.device)
        if self.reasoning_loss == "last_token":
            return dict(q=self.value_head(states[rows, generated.lengths - 1]), states=states)
        q_steps = self.value_head(states)
        return dict(
            q=q_steps[rows, generated.lengths - 1, :-1],
            q_steps=q_steps,
            states=states,
        )


embedding = MultiEmbedding(embeddings={"text": Embedding(pretrained=PRETRAINED)})
backbone = Backbone(residual_dtype=torch.float32, pretrained=PRETRAINED, dtype=torch.float32)
backbone.gradient_checkpointing = GRADIENT_CHECKPOINTING
value_head = nn.Linear(backbone.hidden_dim, 4 + int(REASONING_LOSS == "all_tokens"))
online = ReasoningDqn(
    embedding=embedding,
    backbone=backbone,
    value_head=value_head,
    min_steps=MIN_REASONING_STEPS,
    max_steps=MAX_REASONING_STEPS,
    residual_connections=RESIDUAL_CONNECTIONS,
    reasoning_loss=REASONING_LOSS,
).to(device)
target = online.copy().requires_grad_(False).eval()
if USE_KERNELS:
    backbone.enable_kernels()
    cast(Backbone, target.backbone).enable_kernels()

## Train

Generate the online history, then replay once with online gradients and once
with delayed weights. The delayed model never generates a separate trajectory.

For `all_tokens`, every visited state has one Bellman transition:

`think target = -c + gamma_think * max Q_target(next reasoning state, available actions)`

`environment target = reward + gamma * max Q_target(next observation, first reasoning state, available actions)`

Train think at every continuation and the recorded environment action at the
stopping depth. Mask environment actions before the minimum and think at the cap
in generation and targets. No losses train padding or ungenerated depths.

For `last_token`, only the final state's recorded environment action has a loss:

`target = reward - c * (K - 1) + gamma * max Q_target(next observation, final reasoning state)`

The next observation's final state is used in this mode because it is the only
state with a trained value head. The cost belongs to the current observation's
reasoning and is charged once. Both modes handle terminal discounts independently
and leave the recorded environment rewards unchanged.


In [ ]:
optimizer = MouseOptimizer(
    optimizer={"name": "adamw", "betas": (0.9, 0.95), "eps": 1e-8, "weight_decay": 0.0},
    groups={
        "embedding": {
            "online": online.embedding,
            "lr": LR_EMBEDDING,
            "target": target.embedding,
            "polyak_tau": POLYAK_TAU_EMBEDDING,
        },
        "backbone": {
            "online": online.backbone,
            "lr": LR_BACKBONE,
            "target": target.backbone,
            "polyak_tau": POLYAK_TAU_BACKBONE,
        },
        "value_head": {
            "online": online.value_head,
            "lr": LR_HEADS,
            "target": target.value_head,
            "polyak_tau": POLYAK_TAU_VALUE,
        },
        "reasoner.start_embedding": {
            "online": online.reasoner.start_embedding,
            "lr": LR_HEADS,
            "target": target.reasoner.start_embedding,
            "polyak_tau": POLYAK_TAU_VALUE,
        },
        "reasoner.update_projection": {
            "online": online.reasoner.update_projection,
            "lr": LR_HEADS,
            "target": target.reasoner.update_projection,
            "polyak_tau": POLYAK_TAU_VALUE,
        },
        "reasoner.norm": {
            "online": online.reasoner.norm,
            "lr": LR_BACKBONE,
            "target": target.reasoner.norm,
            "polyak_tau": POLYAK_TAU_BACKBONE,
        },
    },
)

objective = DqnObjective(
    reward=None,
    value=None,
    gate=None,
    temperature=0.0,
    discount=boundary_discount(
        gamma_step=GAMMA,
        gamma_terminated=0.0,
        gamma_truncated=0.0,
    ),
)


training_step = 0


def run_train(
    *, num_steps: int
) -> tuple[torch.Tensor, dict[str, float | torch.Tensor]]:
    """Sample recorded episodes, generate reasoning, and train one batch per step."""
    global training_step
    if num_steps < 1:
        raise ValueError("num_steps must be positive")
    if not 0 < MAX_GRAD_NORM < float("inf"):
        raise ValueError("Invalid max_grad_norm")
    loss: torch.Tensor | None = None
    metrics: dict[str, float | torch.Tensor] = {}
    for _ in range(num_steps):
        tokens, objective_data = loader.next_batch()
        tokens = tokens.to(device)
        objective_data = objective_data.to(device)
        sequence_ids = objective_data["sequence_ids"]
        # Eval mode disables stochastic layers; gradients and optimizer updates remain enabled.
        # Keep it through backward so checkpoint recomputation uses the same mode.
        online.eval()
        target.eval()
        generated = online.generate(
            tokens,
            max_active_requests=MAX_ACTIVE_REQUESTS,
        )
        # Both forward calls consume the same complete generated token sequences.
        lengths = generated.lengths
        out = online(generated)
        with torch.no_grad():
            delayed = target(generated)
            if online.reasoning_loss == "all_tokens":
                delayed_steps = delayed["q_steps"]
                delayed_q = delayed_steps[:, 0].clone()
                if online.reasoner.min_steps > 1:
                    delayed_q[:, :-1] = -torch.inf
                if online.reasoner.max_steps == 1:
                    delayed_q = delayed_q[:, :-1]
            else:
                delayed_q = delayed["q"]
        states, q = out["states"], out["q"]
        step_ids = objective_data["step_ids"]
        next_steps = (step_ids + 1).clamp_max(len(objective_data["action"]) - 1)
        actions = objective_data["action"][next_steps].nan_to_num().long()
        selected = q.gather(-1, actions[:, None]).squeeze(-1)
        _, objective_metrics = objective(
            objective_data=objective_data,
            sequence_ids=sequence_ids,
            step_ids=objective_data["step_ids"],
            predictions=selected,
            backup_value_predictions=delayed_q,
            backup_action_predictions=delayed_q,
        )
        weights, backup = (
            objective_metrics["backup_weight"],
            objective_metrics["backup"],
        )
        if not isinstance(weights, torch.Tensor) or not isinstance(
            backup, torch.Tensor
        ):
            raise TypeError("DQN backup values and weights must be tensors")
        if online.reasoning_loss == "all_tokens":
            loss, _ = value_reasoning_loss(
                min_steps=online.reasoner.min_steps,
                q_values=out["q_steps"], target_q_values=delayed_steps, lengths=lengths,
                environment_actions=actions, environment_targets=backup, weights=weights,
                reasoning_step_cost=REASONING_STEP_COST, gamma_think=GAMMA_THINK,
            )
        else:
            loss, _ = final_reasoning_loss(
                q_values=selected, environment_targets=backup, lengths=lengths,
                weights=weights, reasoning_step_cost=REASONING_STEP_COST,
            )
        optimizer.zero_grad(set_to_none=True)
        loss.backward()
        norm = nn.utils.clip_grad_norm_(
            online.parameters(), MAX_GRAD_NORM, error_if_nonfinite=True
        )
        optimizer.step(training_step)
        training_step += 1
        metrics = {
            "loss": float(loss.detach()),
            "dqn_loss": float(loss.detach()),
            "reasoning_cost": float(REASONING_STEP_COST * ((lengths - 1) * weights).sum()
                                    / weights.sum().clamp_min(1)),
            "grad_norm": float(norm),
            "reasoning_state_rms": float(
                states.detach()[torch.arange(len(states), device=device), lengths - 1]
                .square()
                .mean()
                .sqrt()
            ),
            "reasoning_steps": float(lengths.float().mean()),
        }
    assert loss is not None
    return (loss.detach(), metrics)

## Run

Train on fixed replay, regenerating reasoning for each optimizer update.
The selected loss determines whether every generated state or only the final
state receives value supervision. The residual connection remains independent.


In [ ]:
try:
    for cycle in range(NUM_CYCLES):
        loss, metrics = run_train(num_steps=TRAIN_STEPS)
        print(f"cycle={cycle} train  loss={loss.item():.4f}  {metrics}")
finally:
    loader.close()

## Save to Hugging Face

Run this cell to upload the online weights, tokenizer files and inference settings
together to `MODEL_ID` in your authenticated Hugging Face account. Temporary files
are only used to assemble the upload. Optimizer and delayed-network state are not
included. Evaluate the uploaded repository in the matching reasoning section of
[15 — Inference](15_inference.ipynb).


In [ ]:
with TemporaryDirectory() as directory:
    output = Path(directory)
    save_model(model=online, path=output)
    save_tokenizer(tokenizer=tokenizer, path=output / "tokenizer")
    text_tokenizer.save_pretrained(output / "text_tokenizer")
    (output / "reasoning_settings.json").write_text(
        json.dumps(
            {
                "pretrained": PRETRAINED,
                "environment": {
                    "id": "FrozenLake-v1",
                    "map_name": "4x4",
                    "is_slippery": IS_SLIPPERY,
                    "max_episode_steps": MAX_EPISODE_STEPS,
                },
                "prompt_prefix": task_instructions(),
                "min_steps": online.reasoner.min_steps,
                "max_steps": online.reasoner.max_steps,
                "reasoning_step_cost": REASONING_STEP_COST,
                "gamma_think": GAMMA_THINK,
                "reasoning_loss": online.reasoning_loss,
                "residual_connections": online.reasoner.residual_connections,
                "initial_state": "learned_start_embedding",
                "generation_start_token": GENERATION_START_TOKEN,
                "input_norm": "RMSNorm",
                "update_projection": "rms_norm_linear",
                "norm_eps": online.reasoner.norm.eps,
                "explicit_time_embeddings": False,
                "num_actions": 4,
            },
            indent=2,
        )
        + "\n"
    )
    api = HfApi()
    repository = api.create_repo(
        repo_id=MODEL_ID, repo_type="model", private=False, exist_ok=True
    )
    api.upload_folder(
        repo_id=repository.repo_id,
        repo_type="model",
        folder_path=directory,
        commit_message="Upload reasoning model, tokenizers and inference settings",
    )
print(f"Uploaded checkpoint bundle to {repository}")